# AI4101 · Lab 2 · From Search to Routes: Ekram as a Travelling Salesman Problem

**Work in pairs.** Run the cells from top to bottom with **Shift + Enter**. Cells marked **TODO** are yours to complete.

Before you start: **File → Save a copy in Drive**. Your changes are saved only in your copy.

> The Ekram scenario in this lab is **simulated**. It is inspired by the work of Ekram Al-Jood Association in Makkah. The warehouse, the donors and all numbers are invented for learning.

## Part 1 · The Ekram scenario

One vehicle leaves the Ekram warehouse in **Al-Awali**, visits six donors to collect surplus food, and returns to the warehouse. Each donor is visited **exactly once**.

**The question:** in which order should the vehicle visit the donors so that the total distance is as short as possible?

This is the **Travelling Salesman Problem (TSP)**.

In [ ]:
# The warehouse (index 0) and six donors. Latitude, longitude in degrees. All simulated.
places = [
    ("Warehouse (Al-Awali)",      21.3600, 39.8750),
    ("D1 Restaurant, Al-Aziziyah", 21.4050, 39.8750),
    ("D2 Hotel kitchen, Al-Shawqiyah", 21.3900, 39.8000),
    ("D3 Wedding hall, Al-Zahir", 21.4450, 39.8150),
    ("D4 Bakery, Al-Naseem",      21.3950, 39.8550),
    ("D5 Hotel, Ajyad",           21.4180, 39.8270),
    ("D6 Catering, Al-Kaakiyah",  21.3700, 39.8400),
]
names = [p[0] for p in places]
coords = [(p[1], p[2]) for p in places]
for i, (n, lat, lon) in enumerate(places):
    print(i, n, lat, lon)

In [ ]:
import matplotlib.pyplot as plt

def plot_places(coords, names, route=None, title="Ekram: warehouse and donors"):
    lats = [c[0] for c in coords]; lons = [c[1] for c in coords]
    plt.figure(figsize=(7, 6))
    if route is not None:
        xs = [lons[i] for i in route]; ys = [lats[i] for i in route]
        plt.plot(xs, ys, "-", color="#3D2A6B", linewidth=2, zorder=1)
    plt.scatter(lons[1:], lats[1:], s=80, color="#D4A574", edgecolor="black", zorder=2)
    plt.scatter([lons[0]], [lats[0]], s=160, marker="s", color="#3D2A6B", zorder=3)
    for i, n in enumerate(names):
        plt.annotate(n.split(",")[0], (lons[i], lats[i]), textcoords="offset points", xytext=(6, 6), fontsize=9)
    plt.margins(0.15); plt.xlabel("Longitude"); plt.ylabel("Latitude"); plt.title(title); plt.grid(alpha=0.3)
    plt.show()

plot_places(coords, names)

## Part 2 · Model the problem on paper

Before any more code, write the **five parts** of this search problem in section 1 of your report (Module 3, section 3.1): initial state, actions, transition model, goal test, and action cost.

Then continue.

## Part 3 · The distance matrix

We use the **straight-line distance** ("as the crow flies") between two points on the Earth, in kilometres. Real roads are longer; we come back to this in Lab 3.

The function `straight_line_km` is given. **TODO 1:** fill the matrix `D`, where `D[i][j]` is the distance from place `i` to place `j`.

In [ ]:
import math

def straight_line_km(a, b):
    """Straight-line (great-circle) distance in km between two (lat, lon) points."""
    R = 6371.0
    lat1, lon1 = map(math.radians, a)
    lat2, lon2 = map(math.radians, b)
    h = math.sin((lat2 - lat1) / 2) ** 2 + math.cos(lat1) * math.cos(lat2) * math.sin((lon2 - lon1) / 2) ** 2
    return 2 * R * math.asin(math.sqrt(h))

def distance_matrix(coords):
    n = len(coords)
    D = [[0.0] * n for _ in range(n)]
    # TODO 1: fill D[i][j] with the straight-line distance from coords[i] to coords[j]
    # Hint: two loops, one call to straight_line_km
    return D

D = distance_matrix(coords)
for row in D:
    print(" ".join(f"{x:6.2f}" for x in row))

**Check yourself:** the diagonal must be 0, and `D[i][j]` must equal `D[j][i]`. Why?

## Part 4 · Brute force: try every order

A **route** starts at the warehouse (0), visits every donor once, and returns to 0.

**TODO 2:** complete `route_length`. Then run the brute-force cell. It tries **every** order of the donors and keeps the shortest.

In [ ]:
def route_length(route, D):
    """Total length of a route such as [0, 3, 1, 2, 0]."""
    total = 0.0
    # TODO 2: add the distance of every step: route[0]->route[1], route[1]->route[2], ...
    return total

print(route_length([0, 1, 2, 3, 4, 5, 6, 0], D))

In [ ]:
import itertools, time

def brute_force(D):
    donors = list(range(1, len(D)))
    best_route, best_len, count = None, float("inf"), 0
    for order in itertools.permutations(donors):
        route = [0] + list(order) + [0]
        L = route_length(route, D)
        count += 1
        if L < best_len:
            best_route, best_len = route, L
    return best_route, best_len, count

t0 = time.time()
best_route, best_len, count = brute_force(D)
t_bf = time.time() - t0
print("Routes tried:", count)
print("Best route:", best_route, "->", [names[i].split(" ")[0] for i in best_route])
print(f"Best length: {best_len:.2f} km   Time: {t_bf:.4f} s")
plot_places(coords, names, best_route, "The optimal route (brute force)")

## Part 5 · Nearest neighbour: a greedy method

A quicker idea: **from where you are, always drive to the closest donor you have not visited yet**; when all are visited, go back to the warehouse.

This is a **greedy** method: it makes the choice that looks best right now and never changes it.

**TODO 3:** complete `nearest_neighbour`.

In [ ]:
def nearest_neighbour(D, start=0):
    n = len(D)
    route = [start]
    unvisited = set(range(n)) - {start}
    # TODO 3: while there are unvisited places,
    #         find the unvisited place closest to route[-1], append it, remove it from unvisited
    route.append(start)
    return route

nn_route = nearest_neighbour(D)
nn_len = route_length(nn_route, D)
if best_len == 0 or len(nn_route) < len(D) + 1:
    print("Complete TODO 1, TODO 2 and TODO 3 first, then run this cell again.")
gap = 100 * (nn_len - best_len) / best_len if best_len > 0 else float("nan")
print("Nearest neighbour route:", nn_route)
print(f"Length: {nn_len:.2f} km   Gap to optimal: {gap:.1f}%")
plot_places(coords, names, nn_route, "Nearest neighbour route")

**The gap** tells you how far a route is from the best one: `gap = (your length − optimal length) / optimal length × 100%`. A gap of 0% means the route is optimal.

## Part 6 · How fast does the problem grow?

We now make **random** problems with more donors and time brute force on each. The `seed` makes the random numbers the same every time, so your results can be checked.

Run the cell. It stops at 10 donors, which already takes several seconds.

In [ ]:
import random, math

def random_problem(n_donors, seed):
    rng = random.Random(seed)
    pts = [(21.36, 39.875)] + [(rng.uniform(21.33, 21.46), rng.uniform(39.78, 39.92)) for _ in range(n_donors)]
    return distance_matrix(pts)

results = []
for n in range(4, 11):
    Dn = random_problem(n, seed=2026)
    t0 = time.time(); _, Ln, cnt = brute_force(Dn); t = time.time() - t0
    results.append((n, cnt, t))
    print(f"{n:2d} donors  {cnt:>10,d} routes  {t:8.3f} s")

In [ ]:
ns = [r[0] for r in results]; ts = [r[2] for r in results]
plt.figure(figsize=(7, 4))
plt.plot(ns, ts, "o-", color="#3D2A6B")
plt.yscale("log"); plt.xlabel("Number of donors"); plt.ylabel("Brute-force time (s, log scale)")
plt.title("Brute force grows like n!"); plt.grid(alpha=0.3, which="both"); plt.show()

# Estimate: time per route from the largest run, times n! routes
def human(sec):
    for unit, size in (("years", 365 * 24 * 3600), ("days", 24 * 3600), ("hours", 3600), ("minutes", 60)):
        if sec >= size:
            v = sec / size
            return (f"{v:,.3g}" if v < 1000 else f"{v:,.0f}" if v < 1e6 else f"{v:.1e}") + " " + unit
    return f"{sec:.3g} seconds"

per_route = results[-1][2] / results[-1][1]
for n in (12, 15, 20, 40):
    print(f"{n:2d} donors: about {human(per_route * math.factorial(n))}")

## Part 7 · An academic benchmark: eil51

Researchers test TSP methods on **benchmark** problems whose optimal length is known. We use **eil51** from the TSPLIB library: 51 cities, optimal length **426**.

TSPLIB rounds every distance to the nearest whole number before adding them. The cell below does the same, so your numbers match the published ones.

In [ ]:
import urllib.request

URL = "https://raw.githubusercontent.com/mastqe/tsplib/master/eil51.tsp"
lines = urllib.request.urlopen(URL).read().decode().splitlines()
start = lines.index("NODE_COORD_SECTION") + 1
pts = []
for line in lines[start:]:
    if line.strip() in ("EOF", ""):
        break
    _, x, y = line.split()
    pts.append((float(x), float(y)))

def tsplib_matrix(pts):
    n = len(pts)
    return [[round(math.dist(pts[i], pts[j])) for j in range(n)] for i in range(n)]

E = tsplib_matrix(pts)
OPT = 426
print(len(pts), "cities")

In [ ]:
nn_e = nearest_neighbour(E, start=0)
L = route_length(nn_e, E)
print(f"Nearest neighbour on eil51: {L:.0f}   Gap to optimal (426): {100 * (L - OPT) / OPT:.1f}%")

plt.figure(figsize=(6, 6))
xs = [pts[i][0] for i in nn_e]; ys = [pts[i][1] for i in nn_e]
plt.plot(xs, ys, "-o", color="#3D2A6B", markersize=4)
plt.title(f"eil51: nearest neighbour, length {L:.0f}"); plt.axis("equal"); plt.show()

**Brute force on eil51** would try 50! routes, a number with 65 digits. It is not possible. Lab 3 starts from here: methods that improve a route without trying every order.

## Optional challenge

Nearest neighbour depends on where it starts. Run it from **every** city of eil51 (`start = 0 … 50`) and keep the shortest route. What is the best gap you can reach?

In [ ]:
# Optional: your code here
